# NAVSIM: 自動運転 Pseudo-Simulation パイプライン (EgoStatusMLP)

このノートブックでは、[NAVSIM](https://github.com/autonomousvision/navsim) の EgoStatusMLP ベースラインエージェントを SageMaker AI ML Pipeline で学習・評価します。

### NAVSIM とは

NAVSIM は自動運転車の End-to-End 運転モデルを評価するためのフレームワークです。従来の評価手法には以下の課題がありました。

- **Open-loop 評価**: 記録済みデータに対して予測を行い GT と比較する。高速だが、エラー蓄積や回復動作を評価できない
- **Closed-loop 評価**: シミュレータ内でモデルを走行させる。現実に近いが、計算コストが高い

NAVSIM はこの 2 つのギャップを埋める **Pseudo-Simulation** を提案しています。実データに合成観測を加えることで、Open-loop の効率性を保ちつつ Closed-loop に近い評価精度を実現します。

### EgoStatusMLP

本ノートブックで学習するモデルです。カメラや LiDAR などのセンサー入力を一切使わず、自車の状態 (速度・加速度・走行コマンド) のみから将来 4 秒間の軌跡を予測する軽量な MLP (Multi-Layer Perceptron) です。「センサーなしでどこまでいけるか」の上限を示すベースラインとして位置づけられています。

### コンテナ設計について

SageMaker で使用するコンテナ (Dockerfile) には navsim devkit をインストールしていません。navsim v1.1 は Python 3.9 + `numpy==1.23.4` 等の古いバージョンをピン留めしており、PyTorch DLC (Python 3.11) と依存関係が競合するためです。代わりに、特徴量抽出はこの notebook 上で navsim devkit を使って事前に行い、学習・評価コンテナには前処理済みの npz データだけを渡す設計にしています。詳細は `pipelines/container-navsim-ego-mlp/README.ja.md` を参照してください。

### このノートブックの流れ

1. **パッケージインストール** - numpy 等の依存関係
2. **データセット準備** - navsim clone、特徴量抽出、データ確認
3. **AWS セットアップ** - AWS リソース設定、S3 アップロード、コンテナビルド
4. **学習** - SageMaker Training Job で EgoStatusMLP を学習
5. **評価** - SageMaker Processing Job で ADE / FDE / PDM Score を計算
6. **結果確認** - 評価メトリクスの表示

---
## 1. パッケージインストール

In [ ]:
# numpy をインストール (navsim devkit が後で 1.23.4 にダウングレードする)
%pip install --quiet numpy
%pip install --quiet --force-reinstall matplotlib


---
## 2. データセット準備

EgoStatusMLP の学習には、EgoStatus 特徴量 (8 次元) と将来軌跡のターゲット (8 ポーズ × 3 次元) を含む前処理済みデータが必要です。

以下のセルを実行すると、パイプラインの動作確認用にダミーデータを生成して S3 にアップロードします。事前準備は不要で、そのまま実行してください。S3 に実データがアップロード済みの場合、既存データは上書きされません。

> **Tip**: 実際の走行データで学習したい場合は、JupyterLab ターミナルで `./pipelines/container-navsim-ego-mlp/scripts/prepare_dataset.sh` を実行すると OpenScene データセットから特徴量を抽出できます。

### データ形式

各サンプルは npz ファイルで、以下のキーを含みます。

| キー | Shape | 説明 |
|------|-------|------|
| `features` | `[N, 8]` | EgoStatus (速度 vx, vy、加速度 ax, ay、走行コマンド one-hot 4D) |
| `targets` | `[N, 8, 3]` | GT 軌跡 (x, y, heading) × 8 ポーズ (4 秒間) |

### 2.1 navsim リポジトリの clone

In [ ]:
import os
import subprocess
import numpy as np

CONTAINER_TAG = "container-navsim-ego-mlp"
WORK_DIR = "/tmp/navsim_workspace"
CACHE_DIR = os.path.join(WORK_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

navsim_dir = os.path.join(WORK_DIR, "navsim")
if not os.path.exists(navsim_dir):
    print("Cloning navsim repository (v1.1)...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "v1.1",
         "https://github.com/autonomousvision/navsim.git", navsim_dir],
        check=True,
    )
    print("Clone complete.")
else:
    print(f"Using existing clone: {navsim_dir}")

### 2.2 navsim devkit のインストール

clone した navsim を editable モードでインストールします。これにより `pyquaternion`、`nuplan-devkit` など navsim が依存するパッケージがすべてインストールされます。

`check=False` にしているのは、nuplan-devkit の一部依存で警告が出ることがあるためです。このセルを実行しなかった場合でも、後続の特徴量抽出がダミーデータにフォールバックします。

In [ ]:
subprocess.run(
    ["pip", "install", "--quiet", "-e", navsim_dir],
    check=False,
)
print("navsim devkit installed.")
# navsim devkit が numpy を 1.23.4 にダウングレードするため、
# matplotlib を numpy 1.x 互換バージョンで再インストール
subprocess.run(["pip", "install", "--quiet", "--force-reinstall", "matplotlib"], check=False)


### 2.3 特徴量抽出

`extract_features.py` は以下の処理を行います。

1. navsim の `SceneLoader` でシーンデータを読み込み
2. `EgoStatusFeatureBuilder` で各シーンの EgoStatus 特徴量を計算
3. `TrajectoryTargetBuilder` で将来軌跡の GT を取得
4. train/test に分割して npz 形式で保存

**入力特徴量 (8 次元)**:
- `velocity_x`, `velocity_y` - 自車速度
- `accel_x`, `accel_y` - 自車加速度
- `cmd_left`, `cmd_straight`, `cmd_right`, `cmd_unknown` - 走行コマンド (one-hot)

**ターゲット (8 ポーズ × 3 次元)**:
- 将来 4 秒間の軌跡 (0.5 秒間隔 × 8 ポーズ)
- 各ポーズは `(x, y, heading)` のローカル座標

In [ ]:
result = subprocess.run(
    ["python3", f"../pipelines/{CONTAINER_TAG}/scripts/extract_features.py",
     "--data-root", os.path.join(WORK_DIR, "dataset"),
     "--cache-dir", CACHE_DIR,
     "--navsim-root", navsim_dir,
     "--split", "mini"],
    text=True,
)
if result.returncode != 0:
    print(f"Feature extraction returned non-zero (exit code {result.returncode})")
    print("Generating dummy data for pipeline testing...")
    n = 500
    features = np.random.randn(n, 8).astype(np.float32)
    targets = np.zeros((n, 8, 3), dtype=np.float32)
    for i in range(8):
        targets[:, i, 0] = features[:, 0] * (i + 1) * 0.5
        targets[:, i, 1] = features[:, 1] * (i + 1) * 0.5
    targets += np.random.randn(*targets.shape).astype(np.float32) * 0.1
    np.savez(os.path.join(CACHE_DIR, "train_data.npz"),
             features=features[:400], targets=targets[:400])
    np.savez(os.path.join(CACHE_DIR, "test_data.npz"),
             features=features[400:], targets=targets[400:])
    print("Dummy data created.")

### 2.4 データの確認

抽出したデータの中身を確認します。特徴量の分布と、サンプル軌跡を可視化します。

In [ ]:
import matplotlib.pyplot as plt

train_npz = os.path.join(CACHE_DIR, "train_data.npz")
test_npz = os.path.join(CACHE_DIR, "test_data.npz")

train_data = np.load(train_npz)
test_data = np.load(test_npz)

train_features = train_data["features"]
train_targets = train_data["targets"]
test_features = test_data["features"]
test_targets = test_data["targets"]

print("=" * 60)
print("データセット概要")
print("=" * 60)
print(f"  Train samples:  {train_features.shape[0]}")
print(f"  Test samples:   {test_features.shape[0]}")
print(f"  Feature shape:  {train_features.shape[1]} dims")
print(f"  Target shape:   {train_targets.shape[1]} poses x {train_targets.shape[2]} dims (x, y, heading)")
print()

feature_names = [
    "velocity_x", "velocity_y",
    "accel_x", "accel_y",
    "cmd_left", "cmd_straight", "cmd_right", "cmd_unknown",
]
print("特徴量の統計:")
print(f"  {'Feature':<15s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for i, name in enumerate(feature_names):
    col = train_features[:, i]
    print(f"  {name:<15s} {col.mean():>8.3f} {col.std():>8.3f} {col.min():>8.3f} {col.max():>8.3f}")

In [ ]:
# 特徴量の分布をヒストグラムで表示
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle("EgoStatus Feature Distributions (Train)", fontsize=14)

for i, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    ax.hist(train_features[:, i], bins=30, alpha=0.7, edgecolor="black", linewidth=0.5)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# サンプル軌跡を BEV (Bird's Eye View) で可視化
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Trajectories (BEV, Train)", fontsize=14)

sample_indices = np.random.choice(len(train_targets), size=8, replace=False)
for ax, idx in zip(axes.flat, sample_indices):
    traj = train_targets[idx]
    x = np.concatenate([[0], traj[:, 0]])
    y = np.concatenate([[0], traj[:, 1]])

    ax.plot(y, x, "b-o", markersize=4, linewidth=1.5, label="GT trajectory")
    ax.plot(0, 0, "gs", markersize=8, label="Ego (start)")
    ax.plot(y[-1], x[-1], "r^", markersize=8, label="End")
    ax.set_xlabel("y (m)")
    ax.set_ylabel("x (m, forward)")
    ax.set_title(f"Sample #{idx}", fontsize=10)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

axes[0, 0].legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

---
## 3. AWS セットアップ

AWS リソースの設定、S3 へのデータアップロード、コンテナイメージのビルドを行います。

CloudFormation スタック (`sagemaker-ai-ml-pipeline-stack`) から SageMaker 実行ロールの ARN を自動取得します。スタックが未作成の場合は、先に `infra/scripts/deploy.sh` を実行してください。

In [ ]:
%pip install --quiet "sagemaker>=2.200,<3" boto3

In [ ]:
import json
import logging
import boto3
logging.getLogger("sagemaker.jumpstart").setLevel(logging.CRITICAL)
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.processing import ScriptProcessor

PROJECT_NAME = os.environ.get("PROJECT_NAME", "sagemaker-ai-ml-pipeline")
REGION = boto3.session.Session().region_name or "us-east-1"

session = sagemaker.Session()
account_id = boto3.client("sts").get_caller_identity()["Account"]

cfn = boto3.client("cloudformation", region_name=REGION)
stack = cfn.describe_stacks(StackName=f"{PROJECT_NAME}-stack")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}
role_arn = outputs["SageMakerRoleArn"]

# S3 パス
dataset_bucket = f"{PROJECT_NAME}-dataset-{account_id}-{REGION}"
model_bucket = f"{PROJECT_NAME}-model-{account_id}-{REGION}"
eval_bucket = f"{PROJECT_NAME}-eval-{account_id}-{REGION}"

train_data_uri = f"s3://{dataset_bucket}/{CONTAINER_TAG}/train"
test_data_uri = f"s3://{dataset_bucket}/{CONTAINER_TAG}/test"
model_output_uri = f"s3://{model_bucket}/output"
eval_output_uri = f"s3://{eval_bucket}/output"

ecr_image_uri = (
    f"{account_id}.dkr.ecr.{REGION}.amazonaws.com"
    f"/{PROJECT_NAME}-container:{CONTAINER_TAG}"
)

print(f"Project:       {PROJECT_NAME}")
print(f"Container:     {CONTAINER_TAG}")
print(f"Train data:    {train_data_uri}")
print(f"Test data:     {test_data_uri}")
print(f"ECR image:     {ecr_image_uri}")

### 3.1 S3 にアップロード

`prepare_dataset.sh` で S3 にデータがアップロード済みの場合はスキップされます。
Section 2 で抽出した npz ファイルのみアップロードします。


In [ ]:
s3 = boto3.client("s3", region_name=REGION)

# S3 に prepare_dataset.sh でアップロード済みのデータがあるか確認
existing = s3.list_objects_v2(
    Bucket=dataset_bucket, Prefix=f"{CONTAINER_TAG}/train/", MaxKeys=1,
)
if existing.get("KeyCount", 0) > 0:
    print(f"S3 に学習データが存在します: s3://{dataset_bucket}/{CONTAINER_TAG}/train/")
    print("アップロードをスキップします (prepare_dataset.sh のデータを使用)")
else:
    for npz_file in os.listdir(CACHE_DIR):
        if not npz_file.endswith(".npz"):
            continue
        local_path = os.path.join(CACHE_DIR, npz_file)
        if "train" in npz_file:
            s3_key = f"{CONTAINER_TAG}/train/{npz_file}"
        else:
            s3_key = f"{CONTAINER_TAG}/test/{npz_file}"
        print(f"Uploading {npz_file} -> s3://{dataset_bucket}/{s3_key}")
        s3.upload_file(local_path, dataset_bucket, s3_key)
    print("Upload complete.")


### 3.2 コンテナイメージのビルド & ECR プッシュ

SageMaker Training Job / Processing Job は ECR 上のコンテナイメージを使用します。初回実行時は以下のセルでビルド & プッシュを実行してください。

このコンテナは PyTorch DLC (GPU 版) をベースにした軽量イメージです。navsim devkit はインストールしていません (Python 3.9 依存関係の競合を回避)。

> **Note**: ECR にイメージが既にプッシュ済みの場合は、このセルをスキップできます。

In [ ]:
ecr_client = boto3.client("ecr", region_name=REGION)
image_exists = False
try:
    ecr_client.describe_images(
        repositoryName=f"{PROJECT_NAME}-container",
        imageIds=[{"imageTag": CONTAINER_TAG}],
    )
    image_exists = True
    print("ECR image already exists. Skipping build.")
    print(f"  {ecr_image_uri}")
except ecr_client.exceptions.ImageNotFoundException:
    print("ECR image not found. Building and pushing...")
except Exception as e:
    print(f"ECR check failed ({e}). Attempting build...")

if not image_exists:
    result = subprocess.run(
        ["bash", "../pipelines/scripts/02-build-and-push-container.sh",
         "-c", CONTAINER_TAG, "--auto-approve", PROJECT_NAME],
        text=True,
        env={**os.environ, "AWS_DEFAULT_REGION": REGION},
    )
    if result.returncode != 0:
        print(f"\nBuild failed (exit code {result.returncode}):")
    else:
        print(f"\nContainer pushed to {ecr_image_uri}")

---
## 4. 学習 (Training Job)

EgoStatusMLP モデルを SageMaker Training Job で学習します。

### 処理の流れ

1. SageMaker が S3 の学習データをコンテナの `SM_CHANNEL_TRAIN` にダウンロード
2. `train.py` が npz ファイルから特徴量とターゲットを読み込み
3. EgoStatusMLP を L1 Loss (MAE) で学習。NAVSIM 公式実装と同じ損失関数
4. 学習済みモデル (`model.pth`) を `SM_MODEL_DIR` に保存

### Estimator の設定

- `image_uri`: ECR にプッシュ済みの BYOC イメージ
- `entry_point` / `source_dir`: `train.py` はコンテナに COPY せず、SDK が S3 経由で注入する。スクリプト変更時にコンテナの再ビルドが不要
- `instance_type`: CPU インスタンス (`ml.c7i.xlarge`) を使用。EgoStatusMLP は軽量なので CPU で十分
- `hyperparameters`: SageMaker SDK が `--key value` 形式の CLI 引数に変換して `train.py` に渡す

In [ ]:
from sagemaker.inputs import TrainingInput

# MLflow App ARN の取得 (存在する場合)
mlflow_app_arn = ""
mlflow_app_url = ""
try:
    cfn_client = boto3.client("cloudformation", region_name=REGION)
    resp = cfn_client.describe_stacks(StackName=f"{PROJECT_NAME}-stack")
    for output in resp["Stacks"][0]["Outputs"]:
        if output["OutputKey"] == "MlflowAppArn":
            mlflow_app_arn = output["OutputValue"]
            mlflow_app_url = f"https://{mlflow_app_arn.split('/')[-1]}.mlflow.sagemaker.{REGION}.amazonaws.com"
            break
    if mlflow_app_arn:
        print(f"MLflow App: {mlflow_app_arn}")
except Exception as e:
    print(f"MLflow App not found ({e}). Metrics will not be recorded.")

In [ ]:
estimator = Estimator(
    image_uri=ecr_image_uri,
    entry_point="train.py",
    source_dir=f"../pipelines/{CONTAINER_TAG}",
    role=role_arn,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    output_path=model_output_uri,
    base_job_name=f"{PROJECT_NAME}-navsim-ego-mlp-train",
    environment={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MLFLOW_APP_URL": mlflow_app_url,
        "MODEL_GROUP_NAME": f"{PROJECT_NAME}-navsim-ego-mlp",
    },
    hyperparameters={
        "epochs": 30,
        "batch-size": 32,
        "learning-rate": 0.001,
    },
    metric_definitions=[
        {"Name": "train:loss", "Regex": r"Epoch .* loss: ([0-9.]+)"},
    ],
    sagemaker_session=session,
)

print("Estimator configured.")

### 学習ジョブの実行

`estimator.fit()` で SageMaker Training Job を起動します。`wait=True` で完了まで待機し、ログをリアルタイムで表示します。

In [ ]:
estimator.fit(
    # input_mode: \"FastFile\" = S3 からオンデマンドストリーミング / \"File\" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=train_data_uri, input_mode="FastFile")},
    wait=True,
    logs="All",
)

print(f"\nModel artifact: {estimator.model_data}")

---
## 5. 評価 (Processing Job)

学習済みモデルをテストデータで評価し、PDM Score ベースのメトリクスを計算します。

### 評価メトリクス

| メトリクス | 説明 |
|-----------|------|
| `pdm_score` | ADE ベースの簡易総合スコア (0-1)。低い ADE ほど高スコア |
| `ade` | Average Displacement Error。全タイムステップの平均 L2 距離 (m) |
| `fde` | Final Displacement Error。最終タイムステップの L2 距離 (m) |
| `heading_error` | 予測 heading と GT heading の平均絶対誤差 (rad) |
| `miss_rate` | FDE > 2.0 m のサンプル割合 |

In [ ]:
eval_processor = ScriptProcessor(
    image_uri=ecr_image_uri,
    role=role_arn,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    command=["python3"],
    base_job_name=f"{PROJECT_NAME}-navsim-ego-mlp-eval",
    env={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MLFLOW_APP_URL": mlflow_app_url,
    },
    sagemaker_session=session,
)

eval_processor.run(
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=estimator.model_data,
            destination="/opt/ml/processing/model",
        ),
        sagemaker.processing.ProcessingInput(
            source=test_data_uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=eval_output_uri,
        ),
    ],
    code=f"../pipelines/{CONTAINER_TAG}/evaluate.py",
)

print("Evaluation complete.")

---
## 6. 結果確認

評価結果 (`evaluation.json`) を S3 からダウンロードして表示します。

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    eval_key = "output/evaluation.json"
    local_eval = os.path.join(tmpdir, "evaluation.json")
    try:
        s3.download_file(eval_bucket, eval_key, local_eval)
        with open(local_eval) as f:
            metrics = json.load(f)
        print("=" * 50)
        print("NAVSIM EgoStatusMLP 評価結果")
        print("=" * 50)
        for k, v in metrics.items():
            print(f"  {k:25s}: {v}")
        print("=" * 50)
    except Exception as e:
        print(f"Failed to download evaluation results: {e}")
        print(f"Check S3: s3://{eval_bucket}/output/")

---
## 7. Pipeline 実行 (オプション)

上記の学習・評価を SageMaker Pipeline として一括実行することもできます。

```bash
# ターミナルから実行
./pipelines/scripts/run-pipeline.sh -c container-navsim-ego-mlp
```

In [ ]:
# Pipeline 実行 (CLI 経由)
# !cd .. && ./pipelines/scripts/run-pipeline.sh -c {CONTAINER_TAG}